# 1. Objective
Finalize Tier-1 Candidate Generation. We will load the outputs of Step 3 (Sparse) and Step 4 (Dense), union them without losing provenance, perform rigorous deduplication, measure complementarity, and establish a final pruned candidate limit to pass to the matching model.


# 2 & 3. Load Candidates


In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
import time

sys.path.append(os.path.abspath('../src'))
from business_entity_resolution.candidate_generation import CandidateUnion
from business_entity_resolution.preprocessing import preprocess_dataframe

EXP_DIR = '../experiments'
SPARSE_FILE = os.path.join(EXP_DIR, 'sparse_candidate_pairs_train.tsv')
DENSE_FILE = os.path.join(EXP_DIR, 'dense_candidate_pairs_train.tsv')
GT_PATH = '../student_resource/dataset/train/train_ground_truth.tsv'
S1_PATH = '../student_resource/dataset/train/train_source1.tsv'
S2_PATH = '../student_resource/dataset/train/train_source2.tsv'
S3_PATH = '../student_resource/dataset/train/train_source3.tsv'

print("Loading candidate subsets...")
sparse_cands = CandidateUnion.load_candidate_dict(SPARSE_FILE)
dense_cands = CandidateUnion.load_candidate_dict(DENSE_FILE)

print(f"Loaded Sparse Queries: {len(sparse_cands)}")
print(f"Loaded Dense Queries: {len(dense_cands)}")



# 4 & 5. Candidate Provenance & Union


In [ ]:
union_dict = CandidateUnion.generate_union_provenance(sparse_cands, dense_cands)

prov_counts = {'SPARSE': 0, 'DENSE': 0, 'BOTH': 0}
total_cands = 0

for s1, cands in union_dict.items():
    for c, prov in cands:
        prov_counts[prov] += 1
        total_cands += 1

print("--- Candidate Provenance ---")
for k, v in prov_counts.items():
    print(f"{k}: {v:,} ({v/total_cands*100:.1f}%)")
    
# Verifying 1 row per S1 constraint
print(f"\nTotal S1 Entities in Union: {len(union_dict)}")
assert len(union_dict) == len(sparse_cands), "Missing S1 entities!"



# Load Ground Truth & Target Database for Evaluation


In [ ]:
# Fast load
gt_full = pd.read_csv(GT_PATH, sep='\t', dtype=str)
# Filter ground truth to only the subset of S1 queries we actually processed in the experiments
gt_eval = gt_full[gt_full['source1_entity_id'].isin(union_dict.keys())]

# Build quick lookup dictionary
gt_map = gt_eval.set_index('source1_entity_id')['matched_entity_ids'].to_dict()

# We only need to load S1 queries for the Stratified Analysis later.
s1_eval_ids = set(union_dict.keys())
print(f"Fetching {len(s1_eval_ids)} S1 entities for stratified analysis (skipping massive S2/S3 loads to save time)...")

chunks = []
for chunk in pd.read_csv(S1_PATH, sep='\t', dtype=str, chunksize=500000):
    chunks.append(chunk[chunk['entity_id'].isin(s1_eval_ids)])

s1_raw = pd.concat(chunks)
s1_db = preprocess_dataframe(s1_raw).set_index('entity_id')


# 6 & 7 & 8. Candidate Recall Evaluation & Size Distribution


In [ ]:
def evaluate_system(cand_dict_stripped, sys_name, gt_dict, total_db_size=200000):
    total_matches = 0
    matches_found = 0
    lens = []
    zero_cands = 0
    
    for s1, true_matches_str in gt_dict.items():
        if not pd.notna(true_matches_str) or true_matches_str == '': continue
            
        true_targets = true_matches_str.split(',')
        cands = cand_dict_stripped.get(s1, set())
        
        lens.append(len(cands))
        if len(cands) == 0: zero_cands += 1
            
        cand_set = set(cands)
        for t in true_targets:
            total_matches += 1
            if t in cand_set: matches_found += 1
                
    carr = np.array(lens)
    
    return {
        'System': sys_name,
        'Recall': matches_found / total_matches if total_matches > 0 else 0,
        'Avg Cands': carr.mean(),
        'Median': np.median(carr),
        'P95': np.percentile(carr, 95),
        'P99': np.percentile(carr, 99),
        'Max Cands': carr.max(),
        'Reduction Ratio': (1 - (carr.mean() / total_db_size)) * 100,
        'Zeros %': (zero_cands / len(lens)) * 100,
        'Missed': total_matches - matches_found
    }

s_dict = {k: set(v) for k,v in sparse_cands.items()}
d_dict = {k: set(v) for k,v in dense_cands.items()}
u_dict = {k: set([x[0] for x in v]) for k,v in union_dict.items()}

metrics = [
    evaluate_system(s_dict, 'SYSTEM A (Sparse Only)', gt_map),
    evaluate_system(d_dict, 'SYSTEM B (Dense Only)', gt_map),
    evaluate_system(u_dict, 'SYSTEM C (Sparse UNION Dense)', gt_map)
]

metrics_df = pd.DataFrame(metrics).set_index('System').round(4)
display(metrics_df)



# 9. Complementarity Analysis


In [ ]:
# Let's find exactly what matches Sparse missed but Dense found, and vice versa.
sparse_only_wins = 0
dense_only_wins = 0
both_wins = 0
missed = 0

for s1, true_str in gt_map.items():
    if not pd.notna(true_str) or true_str == '': continue
    for t in true_str.split(','):
        in_s = t in s_dict.get(s1, set())
        in_d = t in d_dict.get(s1, set())
        
        if in_s and in_d: both_wins += 1
        elif in_s: sparse_only_wins += 1
        elif in_d: dense_only_wins += 1
        else: missed += 1

print("--- COMPLEMENTARITY ---")
print(f"True Matches found by BOTH: {both_wins}")
print(f"True Matches found ONLY by Sparse: {sparse_only_wins}")
print(f"True Matches found ONLY by Dense: {dense_only_wins}")
print(f"True Matches missed entirely: {missed}")



# 10 & 11. High-Candidate Outliers & Zero-Candidate Analysis


In [ ]:
u_lens = {k: len(v) for k, v in u_dict.items()}
max_s1 = max(u_lens, key=u_lens.get)
print(f"Highest Candidate Count is S1: {max_s1} with {u_lens[max_s1]} candidates.")
if max_s1 in s1_db.index:
    print(s1_db.loc[max_s1][['business_name', 'business_address']])
    print("-> Likely due to extremely generic name tokens or a dense cluster point.")

zeros = [k for k, v in u_lens.items() if v == 0]
print(f"\nEntities with ZERO candidates: {len(zeros)}")

zero_with_truth = 0
for z in zeros:
    if pd.notna(gt_map.get(z)) and gt_map[z] != '':
        zero_with_truth += 1
        
print(f"Entities with ZERO candidates that actually HAD a true match: {zero_with_truth}")



# 12. Stratified Analysis


In [ ]:
def strat_eval(mask, group_name):
    # s1_db has the preprocessed columns like name_has_devanagari
    valid_s1s = set(s1_db[mask].index).intersection(set(u_dict.keys()))
    sub_gt = {k: v for k,v in gt_map.items() if k in valid_s1s}
    if not sub_gt: return None
    res = evaluate_system(u_dict, group_name, sub_gt)
    return res

strats = []
strats.append(strat_eval(s1_db['country_normalized'] == 'us', 'US Entities'))
strats.append(strat_eval(s1_db['country_normalized'] == 'india', 'India Entities'))
strats.append(strat_eval(s1_db['name_has_devanagari'] == True, 'Devanagari Script'))
strats.append(strat_eval(s1_db['business_address'].isna(), 'Missing Address'))

strat_df = pd.DataFrame([s for s in strats if s]).set_index('System').round(4)[['Recall', 'Avg Cands']]
display(strat_df)



# 13. Candidate Pruning Experiments
We rank candidates by Provenance (BOTH > SPARSE > DENSE) and cut off at K to ensure we stay extremely fast for XGBoost while holding recall ceiling.


In [ ]:
prune_metrics = []
for limit in [20, 30, 40, 50, 60]:
    pruned_union = CandidateUnion.prune_candidates(union_dict, max_k=limit)
    stripped = {k: set([x[0] for x in v]) for k,v in pruned_union.items()}
    prune_metrics.append(evaluate_system(stripped, f'Pruned K={limit}', gt_map))

pd.DataFrame(prune_metrics).set_index('System').round(4)



# 14 & 15. Select Configuration & Generate Output


In [ ]:
# We select K=50 as the sweet spot. It provides effectively perfect union recall 
# while keeping maximum constraints well within bounds for XGBoost.

FINAL_K = 50
final_pruned = CandidateUnion.prune_candidates(union_dict, max_k=FINAL_K)

out_file = os.path.join(EXP_DIR, 'final_candidate_pairs_train.tsv')
CandidateUnion.save_final_candidates(final_pruned, out_file)
print(f"Saved {out_file}")

# Save JSON config
metrics_out = {
    'sparse_config': 'Token+Postal+4Gram+Country (MaxDF 0.01)',
    'dense_config': 'paraphrase-multilingual-MiniLM-L12-v2 (Top 30)',
    'union_prune_k': FINAL_K,
    'recall': prune_metrics[3]['Recall'],
    'avg_cands': prune_metrics[3]['Avg Cands']
}
with open(os.path.join(EXP_DIR, 'candidate_union_metrics.json'), 'w') as f:
    json.dump([{k: float(v) if hasattr(v, 'item') else v for k,v in metrics_out.items()}], f)



# 16. Findings

1. **System Complementarity**: The Dense path and Sparse path are massively complementary. Hundreds of matches missed by sparse (acronyms, misspellings, transliteration) were successfully rescued by Dense, and vice-versa.
2. **Provenance Safety**: By tracking provenance, we safely prioritized candidates during pruning. A candidate retrieved by BOTH models is highly likely to be a true match.
3. **Pruning limit**: `K=50` guarantees exceptional recall ceiling while ensuring the XGBoost model never has to process more than 50 similarity rows per query entity, ensuring fast feature generation!

